# 03 — Make the figures and tables for the paper

**Purpose.** Render every figure the paper uses, from the Gold summary, and display them for
inspection. This notebook produces no numbers of its own — it only draws what notebook 02
computed.

**Inputs.**

- `data/3_gold/tables/recoverability_summary.csv`, written by notebook 02 or by
  `python experiment.py`. If it is missing, run that first.
- `data/3_gold/tables/table1_alternatives_matrix.csv`, which is **hand-authored** — no code
  generates it.
- Figure geometry from `config.FIG_COL_W_IN`, `FIG_FULL_W_IN` and `FIG_DPI`.

**Outputs.** Five PNGs written to `data/3_gold/figures/`, overwriting any existing copies.

**Process.**

1. Put `src/` on the path and import `config` and `plotting`.
2. Render all five figures.
3. Display them inline.
4. Display the hand-authored alternatives matrix.

**A note on figure sizing.** Figures are authored at their **final printed width** — one
IEEE column (3.30 in) or the full page (7.00 in) — and saved without `bbox_inches="tight"`.
A tight bounding box crops the canvas to its content, so whatever places the image has to
scale it back up, silently changing every type size in the figure. `plotting.check_widths()`
asserts each PNG's pixel width matches its intended printed width and runs as part of
`python plotting.py`.

## Setup

`plotting` imports `config`, `estimators` and `synth` itself — the first two figures
illustrate the model rather than plotting the grid, so they regenerate small scenarios on
the fly.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import config
import plotting

## Render the figures

`make_all_figures()` walks the `plotting.FIGURES` registry, which pairs each filename with
its intended printed width and the function that draws it. Adding a figure means adding a
row there, so the registry is the single place the figure set is defined.

What each figure argues:

| Figure | Width | Claim |
|---|---|---|
| `fig1_decomposition` | column | The problem: at one timestamp, events are mixed into the backbone and the meter shows only the total. |
| `fig2_positioning` | column | Where AQF sits against the alternatives, on practicality versus physical representativeness. |
| `fig3_mechanism` | full | *Why* it works: how `p` and `kappa` reshape the across-day distribution, and where each rule cuts it. |
| `fig4_recoverability_map` | full | The centrepiece: `R_F` over the `(p, kappa)` grid for best-fixed-q, AQF and oracle-q, on shared axes and one shared colour scale. |
| `fig5_curves` | column | Accuracy against `kappa`, and the truncation noise floor against `p`. |

Two encoding choices in `fig4` are worth knowing. The colour scale is `log2(R_F)` centred at
zero, so a factor of two too high looks exactly as wrong as a factor of two too low — on a
linear scale it would not. And the fixed-q panel uses `q = 0.3`, the *strongest* fixed
baseline, so the method is shown beating the best available alternative rather than a soft
target.

In [ ]:
plotting.make_all_figures()

## Display the figures

The loop draws its filenames from `plotting.FIGURES` rather than a hardcoded list, so this
cell cannot fall out of step with the figure set — add a figure to the registry and it
appears here automatically.

These render at 600 dpi and at printed size, so they will look large on screen. That is
correct: judge the type size by printing or by viewing at the true physical width, not by
how it fills the notebook.

In [ ]:
from IPython.display import Image, display

# Names come from plotting.FIGURES, so this cell cannot drift from the figure set.
for name, _width_in, _fn in plotting.FIGURES:
    print(name)
    display(Image(filename=str(config.GOLD_FIGURES_DIR / name)))


## The alternatives matrix

`table1_alternatives_matrix.csv` is the odd one out in `data/3_gold/`: it is **hand-authored
positioning, not a computed result**. Nothing in `src/` generates it, and re-running the
pipeline will not change it. It sets out the four candidate approaches — bottom-up appliance
modelling, a simplistic per-timestamp k-factor, the fixed-quantile baseline, and AQF — with
the information each requires and why each was or was not adopted.

It is displayed here so the notebook shows the complete set of artefacts the paper draws on,
but treat it as a document under version control rather than an output.

In [ ]:
import pandas as pd

pd.read_csv(config.GOLD_TABLES_DIR / "table1_alternatives_matrix.csv")

## Conclusion

All five figures are regenerated and at their correct printed widths. Together with the Gold
tables from notebook 02, this is everything the paper needs from this repository — figures,
tables and reproducible numbers, and nothing else. The manuscript itself is drafted outside
this repo; no prose is generated here.

If a figure looks wrong, the fix belongs in `src/plotting.py`, and if a *number* looks
wrong, it belongs in `src/experiment.py` — never in the manuscript. Every value quoted in
the paper must be read from `data/3_gold/tables/`.